# Intellicrack Hexcore Tutorial

A comprehensive, assertion-verified walkthrough of every public method in
`intellicrack_hexcore`, the Rust-backed (PyO3 / maturin) binary editing and
analysis engine.

**Verified specifications (from Rust source):**

| Category | Count | Highlights |
|----------|------:|------------|
| Hash algorithms | 23 | MD5, SHA-1/2/3, BLAKE2, xxHash, SipHash, CRC, FNV, Adler32 |
| Transforms | 23 | XOR, ROT-N, AES-ECB, Base64, zlib, bit/byte ops, masks |
| Built-in templates | 23 | 8 PE, 6 ELF, 4 Mach-O, 3 ZIP, 2 common |
| Encodings | 36 | UTF, ASCII, EBCDIC, ISO-8859-x, Windows-125x, CJK, KOI8 |
| Inspector keys | 40+ | int8–64, float16–64, GUID, IPv4/6, timestamps |

Every code cell includes assertions that **must** pass. A failure indicates a
genuine bug in the hexcore implementation.

## 0. Setup & Import

In [1]:
import intellicrack_hexcore
from intellicrack_hexcore import HexDocument, Bookmark, diff_bytes, diff_files

exports = [x for x in dir(intellicrack_hexcore) if not x.startswith("_")]
print(f"Module: {intellicrack_hexcore.__name__}")
print(f"Exports: {exports}")
assert "HexDocument" in exports
assert "Bookmark" in exports
assert "diff_bytes" in exports
assert "diff_files" in exports
print("Setup OK")

Module: intellicrack_hexcore
Exports: ['Bookmark', 'HexDocument', 'diff_bytes', 'diff_files', 'intellicrack_hexcore']
Setup OK


## 1. Creating Documents

Three constructors:
- `HexDocument()` — empty document (0 bytes)
- `HexDocument.open_bytes(data)` — from a `bytes` object (static method)
- `HexDocument.open(path)` — from a file on disk (static method)

In [ ]:
doc_empty = HexDocument()
assert doc_empty.length() == 0

sample = b"Hello, Hexcore!"
doc = HexDocument.open_bytes(sample)
assert doc.length() == len(sample)

print(f"Empty document: {doc_empty.length()} bytes")
print(f"From bytes:     {doc.length()} bytes")
print("Creating documents OK")

## 2. Reading & Writing

- `read(offset, length)` → `list[int]` (byte values)
- `read_byte(offset)` → `int`
- `write_bytes(offset, data)` — overwrites in-place (recorded for undo)
- `length()` → `int`

In [ ]:
doc = HexDocument.open_bytes(b"ABCDEFGHIJ")

data = doc.read(0, 5)
assert bytes(data) == b"ABCDE"

byte_val = doc.read_byte(0)
assert byte_val == 0x41

assert doc.length() == 10

doc.write_bytes(0, b"XY")
assert bytes(doc.read(0, 2)) == b"XY"
assert bytes(doc.read(2, 3)) == b"CDE"

print(f"Read 5 bytes: {bytes(data)}")
print(f"Single byte at 0: 0x{byte_val:02X}")
print(f"After overwrite: {bytes(doc.read(0, 10))}")
print("Reading & writing OK")

## 3. Insert & Delete

- `insert_bytes(offset, data)` — inserts bytes, shifting subsequent data
- `delete_bytes(offset, length)` — removes bytes, shifting subsequent data

In [ ]:
doc = HexDocument.open_bytes(b"ABCDEF")
assert doc.length() == 6

doc.insert_bytes(3, b"XYZ")
assert doc.length() == 9
assert bytes(doc.read(0, 9)) == b"ABCXYZDEF"

doc.delete_bytes(3, 3)
assert doc.length() == 6
assert bytes(doc.read(0, 6)) == b"ABCDEF"

print("Insert & delete OK")

## 4. Undo / Redo

- `undo()` / `redo()` → `bool` (whether an operation was undone/redone)
- `can_undo()` / `can_redo()` → `bool`
- `is_modified()` → `bool`

In [ ]:
doc = HexDocument.open_bytes(b"ORIGINAL")
assert not doc.is_modified()
assert not doc.can_undo()
assert not doc.can_redo()

doc.write_bytes(0, b"MODIFIED")
assert doc.is_modified()
assert doc.can_undo()
assert bytes(doc.read(0, 8)) == b"MODIFIED"

assert doc.undo()
assert bytes(doc.read(0, 8)) == b"ORIGINAL"
assert doc.can_redo()

assert doc.redo()
assert bytes(doc.read(0, 8)) == b"MODIFIED"

print("Undo / redo OK")

## 5. Save & File Path

- `save(path)` / `save_as(path)` — write document to disk
- `file_path()` → `str | None`

In [ ]:
import tempfile, os

doc = HexDocument.open_bytes(b"Save test data")
assert doc.file_path() is None

with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f:
    tmp_path = f.name

try:
    doc.save(tmp_path)
    doc2 = HexDocument.open(tmp_path)
    assert doc2.length() == doc.length()
    assert bytes(doc2.read(0, doc2.length())) == b"Save test data"
    assert doc2.file_path() is not None
    print(f"Saved to: {doc2.file_path()}")

    copy_path = tmp_path + ".copy"
    doc2.save_as(copy_path)
    assert os.path.exists(copy_path)
    os.unlink(copy_path)
finally:
    os.unlink(tmp_path)

print("Save & file path OK")

## 6. Byte Search

- `search_bytes(pattern, max_results)` → `list[tuple[offset, length]]`
- `search_hex(pattern, max_results)` → same — hex string, `?` per-nibble wildcards

In [ ]:
pe_header = b"MZ" + b"\x00" * 58 + b"\x80\x00\x00\x00" + b"PE\x00\x00" + b"\x00" * 28 + b"MZ_end"
doc = HexDocument.open_bytes(pe_header)

results = doc.search_bytes(b"MZ", 10)
assert len(results) >= 1
assert results[0] == (0, 2)
print(f"search_bytes(b'MZ'): {results}")

hex_results = doc.search_hex("4D 5A", 10)
assert len(hex_results) >= 1
assert hex_results[0] == (0, 2)
print(f"search_hex('4D 5A'): {hex_results}")

wildcard_results = doc.search_hex("4D ?A", 10)
assert len(wildcard_results) >= 1
print(f"search_hex('4D ?A') wildcard: {wildcard_results}")

pe_results = doc.search_bytes(b"PE\x00\x00", 10)
assert len(pe_results) >= 1
print(f"search_bytes(PE sig): {pe_results}")

print("Byte search OK")

## 7. Text Search

- `search_text(text, encoding, case_sensitive, max_results)` → `list[tuple[offset, length]]`
- `search_text_encoded(text, encoding, case_sensitive, max_results)` → same
  (encoding-aware; works with UTF-16, Shift_JIS, etc.)

In [ ]:
text_data = b"Hello World! hello again. HELLO FINAL."
doc = HexDocument.open_bytes(text_data)

case_sensitive = doc.search_text("Hello", "utf-8", True, 10)
assert len(case_sensitive) >= 1
assert case_sensitive[0][0] == 0
print(f"Case-sensitive 'Hello': {case_sensitive}")

case_insensitive = doc.search_text("hello", "utf-8", False, 10)
assert len(case_insensitive) >= 2
print(f"Case-insensitive 'hello': {case_insensitive}")

utf16_payload = "Hello".encode("utf-16-le") + b"\x00" * 10 + "Hello".encode("utf-16-le")
doc2 = HexDocument.open_bytes(utf16_payload)
encoded_results = doc2.search_text_encoded("Hello", "utf-16le", True, 10)
assert len(encoded_results) >= 2
print(f"search_text_encoded UTF-16LE: {encoded_results}")

print("Text search OK")

## 8. Regex Search

- `search_regex(pattern, max_results)` → `list[tuple[offset, length]]`

In [ ]:
data = b"Error: code=404, Error: code=500, Info: code=200"
doc = HexDocument.open_bytes(data)

results = doc.search_regex(r"code=\d+", 10)
assert len(results) == 3
print(f"Regex 'code=\\d+': {results}")
for offset, length in results:
    print(f"  offset={offset}: {bytes(doc.read(offset, length)).decode('ascii')}")

email_data = b"Contact alice@example.com or bob@test.org for info"
doc2 = HexDocument.open_bytes(email_data)
email_results = doc2.search_regex(r"[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+", 10)
assert len(email_results) == 2
print(f"Email regex: {email_results}")

print("Regex search OK")

## 9. Numeric Search

- `search_numeric(value, size, signed, big_endian, alignment, max_results)`
- `search_numeric_float(value, size, big_endian, tolerance, alignment, max_results)`
- `search_numeric_range(value_range, size, signed, big_endian, alignment, max_results)`

All return `list[tuple[offset, length]]`.

In [ ]:
import struct

values = [42, 1000, 42, 65535, 42]
data = struct.pack("<5i", *values)
doc = HexDocument.open_bytes(data)

int_results = doc.search_numeric(42, 4, True, False, 4, 10)
assert len(int_results) == 3
print(f"search_numeric(42, i32 LE): {int_results}")

float_values = [3.14, 2.71, 3.14, 1.41]
float_data = struct.pack("<4f", *float_values)
doc2 = HexDocument.open_bytes(float_data)
float_results = doc2.search_numeric_float(3.14, 4, False, 0.01, 4, 10)
assert len(float_results) == 2
print(f"search_numeric_float(3.14, f32 LE): {float_results}")

range_data = struct.pack("<4i", 10, 100, 200, 300)
doc3 = HexDocument.open_bytes(range_data)
range_results = doc3.search_numeric_range((50, 250), 4, True, False, 4, 10)
assert len(range_results) >= 2
print(f"search_numeric_range(50..250, i32 LE): {range_results}")

print("Numeric search OK")

## 10. Find & Replace

- `replace_bytes(pattern, replacement)` → `int` (count of replacements)

In [ ]:
doc = HexDocument.open_bytes(b"foo bar foo baz foo")
count = doc.replace_bytes(b"foo", b"FOO")
assert count == 3
assert bytes(doc.read(0, doc.length())) == b"FOO bar FOO baz FOO"
print(f"Replaced {count} occurrences: {bytes(doc.read(0, doc.length()))}")

count2 = doc.replace_bytes(b"notfound", b"X")
assert count2 == 0
print(f"No-match replace count: {count2}")

print("Find & replace OK")

## 11. Data Inspector

`inspect_at(offset)` returns a `dict[str, str]` with 40+ interpretations
depending on available bytes:

| Bytes | Keys |
|------:|------|
| 1 | int8, uint8, ascii_char, utf8_char, uleb128, sleb128 |
| 2 | int16_le/be, uint16_le/be, float16_le/be, rgb565, dos_time, dos_date |
| 3 | int24_le/be, uint24_le/be |
| 4 | int32_le/be, uint32_le/be, float32_le/be, rgba8, ipv4, unix_timestamp |
| 6 | int48_le/be, uint48_le/be |
| 8 | int64_le/be, uint64_le/be, float64_le/be, filetime |
| 16 | guid, ipv6 |
| 2+ | wide_string |

In [ ]:
doc = HexDocument.open_bytes(bytes([0x41]) + b"\x00" * 15)
insp = doc.inspect_at(0)
assert isinstance(insp, dict)
assert insp["uint8"] == "65"
assert insp["int8"] == "65"
assert insp["ascii_char"] == "A"
assert insp["uint16_le"] == "65"
print(f"Keys at offset 0: {sorted(insp.keys())}")
print(f"  uint8={insp['uint8']}, ascii_char={insp['ascii_char']}")

ip_data = bytes([192, 168, 1, 1]) + b"\x00" * 12
doc2 = HexDocument.open_bytes(ip_data)
insp2 = doc2.inspect_at(0)
assert insp2["ipv4"] == "192.168.1.1"
print(f"  ipv4={insp2['ipv4']}")

guid_data = bytes([0x01, 0x02, 0x03, 0x04, 0x05, 0x06, 0x07, 0x08,
                   0x09, 0x0A, 0x0B, 0x0C, 0x0D, 0x0E, 0x0F, 0x10])
doc3 = HexDocument.open_bytes(guid_data)
insp3 = doc3.inspect_at(0)
assert insp3["guid"] == "04030201-0605-0807-090a-0b0c0d0e0f10"
print(f"  guid={insp3['guid']}")

ipv6_data = bytes([0x20, 0x01, 0x0d, 0xb8, 0x00, 0x00, 0x00, 0x00,
                   0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x01])
doc4 = HexDocument.open_bytes(ipv6_data)
insp4 = doc4.inspect_at(0)
assert insp4["ipv6"] == "2001:db8:0:0:0:0:0:1"
print(f"  ipv6={insp4['ipv6']}")

import struct
ts_val = 1_704_067_200
ts_data = struct.pack("<I", ts_val) + b"\x00" * 12
doc5 = HexDocument.open_bytes(ts_data)
insp5 = doc5.inspect_at(0)
assert "unix_timestamp" in insp5
assert insp5["unix_timestamp"].startswith("2024-01-01")
print(f"  unix_timestamp={insp5['unix_timestamp']}")

print("Data inspector OK")

## 12. Hashing

23 hash algorithms (42 name variants including aliases), case-insensitive.
All verified against known test vectors from the Rust test suite.

| Category | Algorithms |
|----------|-----------|
| Cryptographic | md5, sha1, sha224, sha256, sha384, sha512, sha3-256, sha3-512, blake2b, blake2s |
| Non-crypto | xxhash32, xxhash64, xxh3, siphash64, siphash128 |
| Checksums | adler32, crc8, crc16, crc32, crc64 |
| FNV | fnv1-32, fnv1-64, fnv1a-32, fnv1a-64 |

In [ ]:
doc_abc = HexDocument.open_bytes(b"abc")
doc_empty = HexDocument.open_bytes(b"")
doc_digits = HexDocument.open_bytes(b"123456789")

assert doc_abc.compute_hash("md5") == "900150983cd24fb0d6963f7d28e17f72"
assert doc_abc.compute_hash("sha1") == "a9993e364706816aba3e25717850c26c9cd0d89d"
assert doc_abc.compute_hash("sha224") == "23097d223405d8228642a477bda255b32aadbce4bda0b3f7e36c9da7"
assert doc_abc.compute_hash("sha256") == "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"
assert doc_abc.compute_hash("sha384").startswith("cb00753f45a35e8b")
assert doc_abc.compute_hash("sha512").startswith("ddaf35a193617aba")
print("Cryptographic hashes: MD5, SHA-1, SHA-224, SHA-256, SHA-384, SHA-512 OK")

assert doc_abc.compute_hash("sha3-256") == doc_abc.compute_hash("sha3_256")
assert doc_abc.compute_hash("sha3-512") == doc_abc.compute_hash("sha3_512")
assert len(doc_abc.compute_hash("blake2b")) == 64
assert len(doc_abc.compute_hash("blake2s")) == 64
print("SHA-3, BLAKE2 OK")

assert doc_empty.compute_hash("xxhash32") == "02cc5d05"
assert doc_empty.compute_hash("xxhash64") == "ef46db3751d8e999"
assert len(doc_abc.compute_hash("xxh3")) == 16
assert len(doc_abc.compute_hash("siphash64")) == 16
assert len(doc_abc.compute_hash("siphash128")) == 32
print("xxHash, SipHash OK")

assert doc_empty.compute_hash("adler32") == "00000001"
assert len(doc_digits.compute_hash("crc8")) == 2
assert len(doc_digits.compute_hash("crc16")) == 4
assert doc_digits.compute_hash("crc32") == "cbf43926"
assert len(doc_digits.compute_hash("crc64")) == 16
print("Adler32, CRC-8/16/32/64 OK")

assert doc_empty.compute_hash("fnv1-32") == "811c9dc5"
assert doc_empty.compute_hash("fnv1-64") == "cbf29ce484222325"
assert doc_empty.compute_hash("fnv1a-32") == "811c9dc5"
assert doc_empty.compute_hash("fnv1a-64") == "cbf29ce484222325"
print("FNV-1/1a OK")

assert doc_abc.compute_hash("SHA256") == doc_abc.compute_hash("sha256")
print("Case-insensitive algorithm names OK")
print("\nAll 23 hash algorithms verified")

In [ ]:
doc = HexDocument.open_bytes(b"Hello World")
full_hash = HexDocument.open_bytes(b"World").compute_hash("sha256")
range_hash = doc.compute_hash_range(6, 11, "sha256")
assert full_hash == range_hash
print(f"compute_hash_range [6:11] matches full-data hash: {range_hash[:32]}...")

crc_doc = HexDocument.open_bytes(b"123456789")
custom_crc = crc_doc.compute_hash_custom_crc(
    (0, 9),
    0x04C11DB7,
    0xFFFFFFFF,
    32,
    (True, True),
    0xFFFFFFFF,
)
assert custom_crc == "cbf43926"
print(f"compute_hash_custom_crc (CRC-32 standard params): {custom_crc}")

print("Hash range & custom CRC OK")

## 13. Entropy Analysis

- `entropy()` → Shannon entropy of entire document (0.0 = uniform, 8.0 = max)
- `entropy_map(block_size)` → per-block entropy values
- `byte_distribution_full()` → 256-element histogram
- `byte_type_distribution()` → `(null, printable, control, high_byte)` counts
- `digram_matrix()` → 65536-element (256×256) bigram frequency matrix
- `content_classification(block_size)` → per-block class (0=null, 1=text, 2=mixed, 3=high-entropy, 4=moderate-high)

In [ ]:
null_block = b"\x00" * 1024
text_block = b"The quick brown fox jumps over the lazy dog. " * 23
prng_block = bytearray(1024)
state = 12345
for i in range(1024):
    state = (state * 1103515245 + 12345) & 0xFFFFFFFF
    prng_block[i] = (state >> 16) & 0xFF

doc_null = HexDocument.open_bytes(null_block)
ent_null = doc_null.entropy()
assert ent_null < 0.01
print(f"Null block entropy:  {ent_null:.4f}")

doc_text = HexDocument.open_bytes(text_block)
ent_text = doc_text.entropy()
assert 3.0 < ent_text < 6.0
print(f"Text block entropy:  {ent_text:.4f}")

doc_prng = HexDocument.open_bytes(bytes(prng_block))
ent_prng = doc_prng.entropy()
assert ent_prng > 6.5
print(f"PRNG block entropy:  {ent_prng:.4f}")

combined = null_block + text_block[:1024] + bytes(prng_block)
doc_combined = HexDocument.open_bytes(combined)

emap = doc_combined.entropy_map(1024)
assert len(emap) == 3
assert emap[0] < 0.01
assert emap[1] > 3.0
assert emap[2] > 6.5
print(f"Entropy map (3 blocks): [{emap[0]:.2f}, {emap[1]:.2f}, {emap[2]:.2f}]")

dist = doc_prng.byte_distribution_full()
assert len(dist) == 256
print(f"Byte distribution: {len(dist)} entries, total={sum(dist)}")

null_count, printable_count, control_count, high_count = doc_text.byte_type_distribution()
assert printable_count > null_count
print(f"Byte types (text): null={null_count}, printable={printable_count}, "
      f"control={control_count}, high={high_count}")

digram = doc_text.digram_matrix()
assert len(digram) == 65536
print(f"Digram matrix: {len(digram)} entries (256x256)")

classes = doc_combined.content_classification(1024)
assert len(classes) == 3
assert classes[0] == 0
assert classes[1] == 1
assert classes[2] in (3, 4)
CLASS_NAMES = {0: "null", 1: "text", 2: "mixed", 3: "high-entropy", 4: "moderate-high"}
print(f"Content classification: {[CLASS_NAMES.get(c, '?') for c in classes]}")

print("Entropy analysis OK")

## 14. Transforms

23 transforms across 7 categories, applied via:
```python
doc.transform_data(name, offset, length, params: dict[str, bytes]) -> list[int]
HexDocument.list_transforms() -> list[tuple[name, category, description]]
```

In [ ]:
transforms = HexDocument.list_transforms()
assert len(transforms) == 23
print(f"Total transforms: {len(transforms)}")
categories: dict[str, list[str]] = {}
for name, category, desc in transforms:
    categories.setdefault(category, []).append(name)
for cat, names in sorted(categories.items()):
    print(f"  [{cat}] {', '.join(names)}")

### 14a. XOR Transforms

In [ ]:
doc = HexDocument.open_bytes(b"Hello World!")

encrypted = doc.transform_data("xor_single", 0, doc.length(), {"key": b"\x42"})
doc2 = HexDocument.open_bytes(bytes(encrypted))
decrypted = doc2.transform_data("xor_single", 0, doc2.length(), {"key": b"\x42"})
assert bytes(decrypted) == b"Hello World!"
print("xor_single roundtrip OK")

enc = doc.transform_data("xor_repeating", 0, doc.length(), {"key": b"KEY"})
doc3 = HexDocument.open_bytes(bytes(enc))
dec = doc3.transform_data("xor_repeating", 0, doc3.length(), {"key": b"KEY"})
assert bytes(dec) == b"Hello World!"
print("xor_repeating roundtrip OK")

doc4 = HexDocument.open_bytes(b"\x00\x00\x00\x00")
rolled = doc4.transform_data("xor_rolling", 0, 4, {"key": b"\x10", "increment": b"\x01"})
assert bytes(rolled) == bytes([0x10, 0x11, 0x12, 0x13])
print(f"xor_rolling: {list(rolled)}")

doc5 = HexDocument.open_bytes(b"\x00\x00\x00")
rolled_default = doc5.transform_data("xor_rolling", 0, 3, {"key": b"\x10"})
assert bytes(rolled_default) == bytes([0x10, 0x11, 0x12])
print("xor_rolling default increment=1 OK")

### 14b. Cipher Transforms

In [ ]:
doc = HexDocument.open_bytes(b"Hello World!!!!!")

rotated = doc.transform_data("rot_n", 0, doc.length(), {"shift": b"\x0d"})
doc2 = HexDocument.open_bytes(bytes(rotated))
unrotated = doc2.transform_data("rot_n", 0, doc2.length(), {"shift": b"\x0d"})
assert bytes(unrotated) == b"Hello World!!!!!"
print(f"ROT13: {bytes(rotated).decode('ascii')} -> roundtrip OK")

key_128 = b"\x00" * 16
aes_data = b"A" * 16
doc3 = HexDocument.open_bytes(aes_data)
encrypted = doc3.transform_data("aes_ecb_encrypt", 0, 16, {"key": key_128})
assert bytes(encrypted) != aes_data
doc4 = HexDocument.open_bytes(bytes(encrypted))
decrypted = doc4.transform_data("aes_ecb_decrypt", 0, 16, {"key": key_128})
assert bytes(decrypted) == aes_data
print("AES-128 ECB roundtrip OK")

key_256 = b"\x01" * 32
doc5 = HexDocument.open_bytes(aes_data)
enc256 = doc5.transform_data("aes_ecb_encrypt", 0, 16, {"key": key_256})
doc6 = HexDocument.open_bytes(bytes(enc256))
dec256 = doc6.transform_data("aes_ecb_decrypt", 0, 16, {"key": key_256})
assert bytes(dec256) == aes_data
print("AES-256 ECB roundtrip OK")

### 14c. Encoding & Compression Transforms

In [ ]:
doc = HexDocument.open_bytes(b"Hello World!")

encoded = doc.transform_data("base64_encode", 0, doc.length(), {})
assert bytes(encoded) == b"SGVsbG8gV29ybGQh"
doc2 = HexDocument.open_bytes(bytes(encoded))
decoded = doc2.transform_data("base64_decode", 0, doc2.length(), {})
assert bytes(decoded) == b"Hello World!"
print(f"Base64: {bytes(encoded).decode('ascii')} -> roundtrip OK")

big_data = b"AAAA" * 256
doc3 = HexDocument.open_bytes(big_data)
compressed = doc3.transform_data("zlib_deflate", 0, doc3.length(), {})
assert len(compressed) < len(big_data)
doc4 = HexDocument.open_bytes(bytes(compressed))
decompressed = doc4.transform_data("zlib_inflate", 0, doc4.length(), {})
assert bytes(decompressed) == big_data
ratio = 100 * len(compressed) / len(big_data)
print(f"Zlib: {len(big_data)} -> {len(compressed)} bytes ({ratio:.1f}%) -> roundtrip OK")

### 14d. Bit Operation Transforms

In [ ]:
doc = HexDocument.open_bytes(bytes([0b00000001]))
shifted = doc.transform_data("bit_shift_left", 0, 1, {"count": b"\x02"})
assert list(shifted) == [0b00000100]
print(f"bit_shift_left(0x01, 2): 0b{shifted[0]:08b}")

doc2 = HexDocument.open_bytes(bytes([0b10000000]))
shifted_r = doc2.transform_data("bit_shift_right", 0, 1, {"count": b"\x03"})
assert list(shifted_r) == [0b00010000]
print(f"bit_shift_right(0x80, 3): 0b{shifted_r[0]:08b}")

doc3 = HexDocument.open_bytes(bytes([0b10000001]))
rotated = doc3.transform_data("bit_rotate_left", 0, 1, {"count": b"\x01"})
assert list(rotated) == [0b00000011]
print(f"bit_rotate_left(0x81, 1): 0b{rotated[0]:08b}")

doc4 = HexDocument.open_bytes(bytes([0b10000001]))
rotated_r = doc4.transform_data("bit_rotate_right", 0, 1, {"count": b"\x01"})
assert list(rotated_r) == [0b11000000]
print(f"bit_rotate_right(0x81, 1): 0b{rotated_r[0]:08b}")

doc5 = HexDocument.open_bytes(b"Test")
inverted = doc5.transform_data("bit_invert", 0, 4, {})
doc6 = HexDocument.open_bytes(bytes(inverted))
restored = doc6.transform_data("bit_invert", 0, 4, {})
assert bytes(restored) == b"Test"
print("bit_invert roundtrip OK")

### 14e. Byte Operation Transforms

In [ ]:
doc = HexDocument.open_bytes(b"ABCDE")
reversed_data = doc.transform_data("byte_reverse", 0, 5, {})
assert bytes(reversed_data) == b"EDCBA"
print(f"byte_reverse: {bytes(reversed_data)}")

doc2 = HexDocument.open_bytes(bytes([0x01, 0x02, 0x03, 0x04]))
swapped = doc2.transform_data("byte_swap_16", 0, 4, {})
assert list(swapped) == [0x02, 0x01, 0x04, 0x03]
print(f"byte_swap_16: {swapped}")

doc3 = HexDocument.open_bytes(bytes([0x01, 0x02, 0x03, 0x04]))
swapped32 = doc3.transform_data("byte_swap_32", 0, 4, {})
assert list(swapped32) == [0x04, 0x03, 0x02, 0x01]
print(f"byte_swap_32: {swapped32}")

doc4 = HexDocument.open_bytes(bytes(range(8)))
swapped64 = doc4.transform_data("byte_swap_64", 0, 8, {})
assert list(swapped64) == [7, 6, 5, 4, 3, 2, 1, 0]
print(f"byte_swap_64: {swapped64}")

doc5 = HexDocument.open_bytes(bytes([0x41, 0x00, 0x42, 0x00, 0x43]))
cleaned = doc5.transform_data("remove_nulls", 0, 5, {})
assert bytes(cleaned) == b"ABC"
print(f"remove_nulls: {bytes(cleaned)}")

### 14f. Mask Transforms

In [ ]:
doc = HexDocument.open_bytes(bytes([0xFF, 0xFF, 0xFF, 0xFF]))
anded = doc.transform_data("mask_and", 0, 4, {"pattern": b"\x0F"})
assert list(anded) == [0x0F, 0x0F, 0x0F, 0x0F]
print(f"mask_and(0xFF, 0x0F): {[f'0x{b:02X}' for b in anded]}")

doc2 = HexDocument.open_bytes(bytes([0x00, 0x00, 0x00, 0x00]))
ored = doc2.transform_data("mask_or", 0, 4, {"pattern": b"\xF0"})
assert list(ored) == [0xF0, 0xF0, 0xF0, 0xF0]
print(f"mask_or(0x00, 0xF0): {[f'0x{b:02X}' for b in ored]}")

doc3 = HexDocument.open_bytes(b"AAAA")
xored = doc3.transform_data("mask_xor", 0, 4, {"pattern": b"\x20"})
assert bytes(xored) == b"aaaa"
print(f"mask_xor('AAAA', 0x20): {bytes(xored)}")

print("All 23 transforms verified")

## 15. Encodings

36 encodings via:
- `HexDocument.list_encodings()` → `list[tuple[name, description]]`
- `doc.decode_text(offset, length, encoding)` → `str`
- `HexDocument.encode_text_to_bytes(text, encoding)` → `list[int]`
- `doc.search_text_encoded(text, encoding, case_sensitive, max_results)`

In [ ]:
encodings = HexDocument.list_encodings()
assert len(encodings) == 36
print(f"Supported encodings: {len(encodings)}")
for name, desc in encodings:
    print(f"  {name}: {desc}")

In [ ]:
test_text = "Hello, World!"

encoded = HexDocument.encode_text_to_bytes(test_text, "utf-8")
doc = HexDocument.open_bytes(bytes(encoded))
decoded = doc.decode_text(0, len(encoded), "utf-8")
assert decoded == test_text
print(f"UTF-8 roundtrip OK ({len(encoded)} bytes)")

encoded_u16 = HexDocument.encode_text_to_bytes(test_text, "utf-16le")
doc2 = HexDocument.open_bytes(bytes(encoded_u16))
decoded_u16 = doc2.decode_text(0, len(encoded_u16), "utf-16le")
assert decoded_u16 == test_text
print(f"UTF-16LE roundtrip OK ({len(encoded_u16)} bytes)")

encoded_ebc = HexDocument.encode_text_to_bytes("Hello World 0123", "ebcdic")
doc3 = HexDocument.open_bytes(bytes(encoded_ebc))
decoded_ebc = doc3.decode_text(0, len(encoded_ebc), "ebcdic")
assert decoded_ebc == "Hello World 0123"
print(f"EBCDIC roundtrip OK ({len(encoded_ebc)} bytes)")

encoded_1252 = HexDocument.encode_text_to_bytes("caf\u00e9", "windows-1252")
doc4 = HexDocument.open_bytes(bytes(encoded_1252))
decoded_1252 = doc4.decode_text(0, len(encoded_1252), "windows-1252")
assert decoded_1252 == "caf\u00e9"
print(f"Windows-1252 roundtrip OK")

encoded_sjis = HexDocument.encode_text_to_bytes("Hello", "shift_jis")
doc5 = HexDocument.open_bytes(bytes(encoded_sjis))
decoded_sjis = doc5.decode_text(0, len(encoded_sjis), "shift_jis")
assert decoded_sjis == "Hello"
print(f"Shift_JIS roundtrip OK")

utf16_payload = bytes(HexDocument.encode_text_to_bytes("Hello World Hello", "utf-16le"))
doc6 = HexDocument.open_bytes(utf16_payload)
results = doc6.search_text_encoded("Hello", "utf-16le", True, 10)
assert len(results) == 2
print(f"search_text_encoded UTF-16LE: {results}")

print("Encodings OK")

## 16. Binary Templates

23 built-in templates:
- **PE** (8): IMAGE_DOS_HEADER, IMAGE_FILE_HEADER, IMAGE_OPTIONAL_HEADER32/64,
  IMAGE_SECTION_HEADER, IMAGE_DATA_DIRECTORY, IMAGE_IMPORT_DESCRIPTOR,
  IMAGE_EXPORT_DIRECTORY
- **ELF** (6): Elf32/64_Ehdr, Elf32/64_Phdr, Elf32/64_Shdr
- **Mach-O** (4): MACH_HEADER, MACH_HEADER_64, LOAD_COMMAND, SEGMENT_COMMAND_64
- **ZIP** (3): ZIP_LOCAL_FILE_HEADER, ZIP_CENTRAL_DIRECTORY,
  ZIP_END_OF_CENTRAL_DIRECTORY
- **Common** (2): GUID, FILETIME

In [ ]:
doc = HexDocument.open_bytes(b"\x00" * 16)
templates = doc.list_templates()
print(f"Built-in templates: {len(templates)}")

expected = [
    "IMAGE_DOS_HEADER", "IMAGE_FILE_HEADER",
    "IMAGE_OPTIONAL_HEADER32", "IMAGE_OPTIONAL_HEADER64",
    "IMAGE_SECTION_HEADER", "IMAGE_DATA_DIRECTORY",
    "IMAGE_IMPORT_DESCRIPTOR", "IMAGE_EXPORT_DIRECTORY",
    "Elf32_Ehdr", "Elf64_Ehdr", "Elf32_Phdr", "Elf64_Phdr",
    "Elf32_Shdr", "Elf64_Shdr",
    "MACH_HEADER", "MACH_HEADER_64", "LOAD_COMMAND", "SEGMENT_COMMAND_64",
    "ZIP_LOCAL_FILE_HEADER", "ZIP_CENTRAL_DIRECTORY", "ZIP_END_OF_CENTRAL_DIRECTORY",
    "GUID", "FILETIME",
]
template_names = [name for name, _ in templates]
for t in expected:
    assert t in template_names, f"Missing template: {t}"
print(f"All {len(expected)} expected templates present")

detailed = doc.list_templates_detailed()
for name, desc, category, field_count in detailed:
    print(f"  {name} [{category}]: {field_count} fields - {desc}")

In [ ]:
pe_data = bytearray(128)
pe_data[0:2] = b"MZ"
pe_data[60:64] = (0x80).to_bytes(4, "little")

doc = HexDocument.open_bytes(bytes(pe_data))
fields = doc.apply_template("IMAGE_DOS_HEADER", 0)
assert len(fields) > 0
assert fields[0]["name"] == "e_magic"
print(f"IMAGE_DOS_HEADER: {len(fields)} fields")
for f in fields[:5]:
    print(f"  {f['name']}: {f['display_value']} (offset={f['offset']}, size={f['size']})")

elf_data = bytearray(64)
elf_data[0:4] = b"\x7fELF"
elf_data[4] = 2
elf_data[5] = 1

doc2 = HexDocument.open_bytes(bytes(elf_data))
elf_fields = doc2.apply_template("Elf64_Ehdr", 0)
assert len(elf_fields) > 0
print(f"\nElf64_Ehdr: {len(elf_fields)} fields")
for f in elf_fields[:5]:
    print(f"  {f['name']}: {f['display_value']}")

zip_data = bytearray(30)
zip_data[0:4] = b"PK\x03\x04"
zip_data[4:6] = (20).to_bytes(2, "little")

doc3 = HexDocument.open_bytes(bytes(zip_data))
zip_fields = doc3.apply_template("ZIP_LOCAL_FILE_HEADER", 0)
assert len(zip_fields) > 0
print(f"\nZIP_LOCAL_FILE_HEADER: {len(zip_fields)} fields")
for f in zip_fields[:5]:
    print(f"  {f['name']}: {f['display_value']}")

macho_data = bytearray(32)
macho_data[0:4] = b"\xCF\xFA\xED\xFE"
macho_data[4:8] = (0x01000007).to_bytes(4, "little")

doc4 = HexDocument.open_bytes(bytes(macho_data))
macho_fields = doc4.apply_template("MACH_HEADER_64", 0)
assert len(macho_fields) > 0
print(f"\nMACH_HEADER_64: {len(macho_fields)} fields")
for f in macho_fields[:5]:
    print(f"  {f['name']}: {f['display_value']}")

print("\nTemplate application OK")

In [ ]:
import json

custom_template = {
    "name": "CUSTOM_HEADER",
    "description": "Custom binary header for tutorial",
    "default_endianness": "little",
    "fields": [
        {"name": "magic", "field_type": {"type": "UInt32"}, "description": "Magic number"},
        {"name": "version", "field_type": {"type": "UInt16"}, "description": "Version"},
        {"name": "flags", "field_type": {"type": "UInt16"}, "description": "Flags"},
        {"name": "name", "field_type": {"type": "FixedString", "params": 8}, "description": "Name"},
    ],
}

doc = HexDocument.open_bytes(b"\xDE\xAD\xBE\xEF\x01\x00\xFF\x00TestName")
name = doc.register_json_template(json.dumps(custom_template))
assert name == "CUSTOM_HEADER"

fields = doc.apply_template("CUSTOM_HEADER", 0)
assert len(fields) == 4
assert fields[0]["name"] == "magic"
print(f"Custom template '{name}': {len(fields)} fields")
for f in fields:
    print(f"  {f['name']}: {f['display_value']}")

exported_json = doc.export_template_json("CUSTOM_HEADER")
assert "CUSTOM_HEADER" in exported_json
print(f"\nExported JSON: {len(exported_json)} chars")

assert doc.remove_template("CUSTOM_HEADER")
assert not doc.remove_template("CUSTOM_HEADER")
print("Custom template register/export/remove OK")

## 17. Bookmarks

- `add_bookmark(offset, length, label, color)` → `int` (index)
- `list_bookmarks()` → `list[tuple[offset, length, label, color]]`
- `remove_bookmark(index)` → `bool`

In [ ]:
doc = HexDocument.open_bytes(b"ABCDEFGHIJKLMNOP")

idx0 = doc.add_bookmark(0, 4, "Header", "#FF0000")
idx1 = doc.add_bookmark(4, 4, "Data", "#00FF00")
idx2 = doc.add_bookmark(8, 4, "Footer", "#0000FF")
assert idx0 == 0
assert idx1 == 1
assert idx2 == 2

bookmarks = doc.list_bookmarks()
assert len(bookmarks) == 3
assert bookmarks[0] == (0, 4, "Header", "#FF0000")
assert bookmarks[1] == (4, 4, "Data", "#00FF00")
assert bookmarks[2] == (8, 4, "Footer", "#0000FF")
print(f"Bookmarks: {bookmarks}")

assert doc.remove_bookmark(1)
bookmarks = doc.list_bookmarks()
assert len(bookmarks) == 2
print(f"After removing index 1: {bookmarks}")

assert not doc.remove_bookmark(99)
print("Bookmarks OK")

## 18. Patch Export / Import

- `get_patches()` → raw overwrite records `list[tuple[offset, bytes]]`
- `export_patches_ips()` → IPS format bytes (PATCH header, EOF footer)
- `export_patches_ips32()` → IPS32 format (IPS32 header, EEOF footer)
- `import_patches_ips(data)` → number of patches applied

In [ ]:
original = b"ABCDEFGHIJKLMNOP"
doc = HexDocument.open_bytes(original)
doc.write_bytes(0, b"XY")
doc.write_bytes(8, b"ZZ")

patches = doc.get_patches()
assert len(patches) >= 2
print(f"Raw patches: {patches}")

ips_data = doc.export_patches_ips()
assert bytes(ips_data[:5]) == b"PATCH"
assert bytes(ips_data[-3:]) == b"EOF"
print(f"IPS export: {len(ips_data)} bytes")

ips32_data = doc.export_patches_ips32()
assert bytes(ips32_data[:5]) == b"IPS32"
assert bytes(ips32_data[-4:]) == b"EEOF"
print(f"IPS32 export: {len(ips32_data)} bytes")

doc2 = HexDocument.open_bytes(original)
count = doc2.import_patches_ips(bytes(ips_data))
assert count >= 2
patched = bytes(doc2.read(0, 16))
assert patched[:2] == b"XY"
assert patched[8:10] == b"ZZ"
print(f"IPS import: {count} patches, result: {patched}")

print("Patch export/import OK")

## 19. Binary Diff

Module-level functions:
- `diff_bytes(data_a, data_b)` → `dict` with `total_differences`, `files_identical`, `regions`
- `diff_files(path_a, path_b)` → same

In [ ]:
result = diff_bytes(b"Hello World!", b"Hello Brave World!")
assert not result["files_identical"]
assert result["total_differences"] > 0
print(f"diff_bytes:")
print(f"  identical: {result['files_identical']}")
print(f"  total_differences: {result['total_differences']}")
for r in result["regions"]:
    print(f"  type={r['diff_type']}, offset_a={r['offset_a']}, "
          f"offset_b={r['offset_b']}, len={r['length']}")

result2 = diff_bytes(b"same", b"same")
assert result2["files_identical"]
assert result2["total_differences"] == 0
print(f"\nIdentical: files_identical={result2['files_identical']}")

import tempfile, os
with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f1:
    f1.write(b"File content A")
    path1 = f1.name
with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as f2:
    f2.write(b"File content B")
    path2 = f2.name
try:
    file_result = diff_files(path1, path2)
    assert not file_result["files_identical"]
    print(f"\ndiff_files: identical={file_result['files_identical']}, "
          f"diffs={file_result['total_differences']}")
finally:
    os.unlink(path1)
    os.unlink(path2)

print("Binary diff OK")

## 20. Byte Statistics

`byte_statistics()` → `list[tuple[byte_value, count]]` (256 entries)

In [ ]:
doc = HexDocument.open_bytes(b"AAABBC")
stats = doc.byte_statistics()
assert len(stats) == 256

stats_dict = {byte_val: count for byte_val, count in stats}
assert stats_dict[0x41] == 3
assert stats_dict[0x42] == 2
assert stats_dict[0x43] == 1
assert stats_dict[0x00] == 0

non_zero = [(bv, c) for bv, c in stats if c > 0]
print(f"Byte statistics for b'AAABBC':")
for bv, c in non_zero:
    print(f"  0x{bv:02X} ('{chr(bv)}'): {c}")

total = sum(c for _, c in stats)
assert total == 6
print(f"Total bytes: {total}")
print("Byte statistics OK")

## 21. Error Handling

Hexcore raises Python exceptions for invalid operations. This section verifies
that all expected error paths produce the correct exception types.

In [ ]:
def expect_error(fn, description):
    try:
        fn()
        print(f"  UNEXPECTED: {description} did NOT raise")
        return False
    except Exception as e:
        print(f"  {description}: {type(e).__name__}")
        return True

doc = HexDocument.open_bytes(b"ABCD")

assert expect_error(lambda: doc.read(1000, 1), "read beyond end")
assert expect_error(lambda: doc.write_bytes(1000, b"X"), "write beyond end")
assert expect_error(lambda: doc.insert_bytes(1000, b"X"), "insert beyond end")
assert expect_error(lambda: doc.delete_bytes(1000, 1), "delete beyond end")
assert expect_error(lambda: doc.compute_hash("nonexistent"), "invalid hash algorithm")
assert expect_error(
    lambda: doc.transform_data("nonexistent", 0, 4, {}), "invalid transform")
assert expect_error(lambda: doc.apply_template("NONEXISTENT", 0), "invalid template")
assert expect_error(lambda: doc.decode_text(0, 4, "nonexistent_enc"), "invalid encoding")
assert expect_error(
    lambda: doc.transform_data("aes_ecb_encrypt", 0, 4, {"key": b"\x00" * 7}),
    "AES with invalid key size")
assert expect_error(
    lambda: doc.transform_data("xor_single", 0, 4, {}), "missing xor key")
assert expect_error(
    lambda: doc.compute_hash_custom_crc((0, 4), 0x04C11DB7, 0, 12, (False, False), 0),
    "invalid CRC width")
assert expect_error(
    lambda: doc.compute_hash_range(10, 5, "md5"), "invalid hash range")
assert expect_error(
    lambda: HexDocument.open("/nonexistent/path/file.bin"), "open nonexistent file")

print("\nAll error cases handled correctly")

## Summary

In [ ]:
print("=" * 60)
print("HEXCORE TUTORIAL COMPLETE")
print("=" * 60)
print("All sections passed:")
print("  Document I/O (new, open, open_bytes, save, save_as)")
print("  Read / Write / Insert / Delete")
print("  Undo / Redo")
print("  Byte & hex search (with wildcards)")
print("  Text & encoded-text search")
print("  Regex search")
print("  Numeric search (int, float, range)")
print("  Find & replace")
print("  Data inspector (40+ interpretation keys)")
print("  23 hash algorithms with test vectors")
print("  Hash range & custom CRC")
print("  Entropy analysis (6 functions)")
print("  23 transforms across 7 categories")
print("  36 encodings with roundtrip verification")
print("  23 built-in templates (PE / ELF / ZIP / Mach-O / common)")
print("  Custom JSON templates")
print("  Bookmarks")
print("  IPS / IPS32 patch export & import")
print("  Binary diff (bytes & files)")
print("  Byte statistics")
print("  Error handling")